In [9]:
import pandas as pd
import torchio as tio
from datasetgenerator import *
from params import *
from monai.networks.nets import UNet
import matplotlib.pyplot as plt
import torch
import nibabel as nib

In [10]:
def monai_unet_model(dropout=0, in_channels=1):
    return UNet(
        spatial_dims=3,
        in_channels=in_channels,
        out_channels=1,
        dropout=dropout,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2
    )
    
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

def show_max_mask_slices(images, mask, subj, alpha):
    """
    Show the maximum content slice for an image, its corresponding mask,
    and the mask as a red overlay on the image.
    """
    # Create a custom colormap for the overlay (red mask)
    red_cmap = ListedColormap(['none', 'red'])

    # Number of channels
    nchannel = len(images)
    
    fig, axes = plt.subplots(1, nchannel+2, figsize=(15, 5))  # Three subplots side by side
    print(f'Shape of image and mask: {images[0].shape}, {mask.shape}')
    
    # Find the slice index with the maximum mask content
    slice_sums = mask.sum(axis=(0, 1))  # Sum over x and y axes to get per-slice content
    max_slice_index = slice_sums.argmax()  # Find the index of the slice with the most mask content
    
    # Plot the image at the slice with maximum mask content
    for i, image in enumerate(images):
        array = image.get_fdata()
        axes[i].imshow(array[:, :, max_slice_index], cmap='gray')
        axes[i].set_title(f'Image Slice {max_slice_index} for subj {subj}')
        axes[i].axis('off')
    
    # Plot the mask at the slice with maximum mask content
    i = i + 1
    axes[i].imshow(mask[:, :, max_slice_index], cmap='gray')
    axes[i].set_title(f'Mask Slice {max_slice_index} for subj {subj}')
    axes[i].axis('off')

    # Plot the overlay of mask on the image
    i = i + 1
    array = images[0].get_fdata()
    axes[i].imshow(array[:, :, max_slice_index], cmap='gray')
    axes[i].imshow(mask[:, :, max_slice_index], cmap=red_cmap, alpha=alpha)  # Add transparency for mask
    axes[i].set_title(f'Red Overlay for subj {subj}')
    axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# Define preprocessing function for multiple channels
def preprocess_images(image_paths, img_size):
    """
    Preprocess multiple images and stack them into channels.
    
    Args:
        image_paths (list of str): List of image paths to preprocess.
        img_size (int): Target image size.
    
    Returns:
        numpy.ndarray: Stacked array of preprocessed images with shape (channels, depth, height, width).
        nib.Nifti1Image: Original image object (for the first image).
    """
    transform = tio.Compose([
        tio.ZNormalization(),
        tio.CropOrPad(target_shape=(img_size, img_size, img_size))
    ])
    processed_images = []
    original_images = []
    
    for path in image_paths:
        subject = tio.Subject(image=tio.ScalarImage(path))
        transformed = transform(subject)
        processed_image = transformed['image'][tio.DATA].squeeze(0)
        processed_images.append(processed_image)
        #print(processed_image.shape)
        #aa
        
        # Save the original image shape from the first image
        original_images.append(nib.load(path))
    print(f'Shape of original image: {original_images[0].shape}')
        
    # Stack along the channel dimension
    stacked_images = np.stack(processed_images, axis=0)  # Shape: (channels, depth, height, width)
    return stacked_images, original_images


In [11]:
# Load the dataframe
df = pd.read_csv(pathlistvalid, sep=';').groupby("subj", as_index=False).first()

# Version of model
version = 'v0.1'

# The model to use
#state = 'single'
state = 'multi'
if state == 'single':
    basename = f'UNet-Monai-VIBE-{version}'
    in_channels = 1
    imgpath_list = ['imgpath1']
    prepath_list = ['unregistered']
    imgname_list = ['vibe2min.nii.gz']
    require = ['pathvibe2minDicom']
elif state == 'multi':
    basename = f'UNet-Monai-VIBE-T2-ADC-{version}'
    in_channels = 3
    imgpath_list = ['imgpath1', 'imgpath2', 'imgpath3']
    prepath_list = ['unregistered','registered','registered']
    imgname_list = ['vibe2min.nii.gz','T2-2-vibe2min-header.nii.gz','ADC-2-vibe2min-header.nii.gz']
    require = ['pathvibe2minDicom', 'pathT2Dicom', 'pathADCDicom']

# Prepare dataframe and find manual masks
df = datasetgenerator(df, require)
#df = datasetgenerator(df, ['pathT2Dicom','pathADCDicom'])

# Assign the filename name
#df.loc[df.dataset == 'ML', 'pathmask'] = 'PREDICTED_VIBE_MONAI_' + version + '.nii.gz'

# Only for this subject
#df = df[df.subj == 129].reset_index(drop=True)

# Remove those with mask, since they are in the training set
df = df[df.dataset == 'ML'].reset_index(drop=True)


# Make full image path
for fn in imgpath_list:
    df[fn] = ""
for i, subj in enumerate(df.subj):
    for j, fn in enumerate(imgpath_list):
        df.loc[i, fn] = os.path.join(prepathnifti,  'EC' + str(subj).zfill(3), prepath_list[j], imgname_list[j])
        
# Keep only those
cols_to_keep = ['subj','pathvibe2minDicom'] + imgpath_list
df = df[cols_to_keep]

# Reset index
df = df.reset_index(drop=True)
print(f'Antall rekker: {len(df)}')
df.tail(5)

579 datasett tilfredsstiller betingelsene
Antall rekker: 306


,subj,pathvibe2minDicom,imgpath1,imgpath2,imgpath3
301,545,22_t1_vibe_dixon_tra_p2__2min_W,/raid/erlend/GynKreft/Data-EC/Nifti/EC545/unre...,/raid/erlend/GynKreft/Data-EC/Nifti/EC545/regi...,/raid/erlend/GynKreft/Data-EC/Nifti/EC545/regi...
302,546,9_fl3d_vibe_tra__2mm_2min,/raid/erlend/GynKreft/Data-EC/Nifti/EC546/unre...,/raid/erlend/GynKreft/Data-EC/Nifti/EC546/regi...,/raid/erlend/GynKreft/Data-EC/Nifti/EC546/regi...
303,548,12_fl3d_vibe_tra__2mm_2min,/raid/erlend/GynKreft/Data-EC/Nifti/EC548/unre...,/raid/erlend/GynKreft/Data-EC/Nifti/EC548/regi...,/raid/erlend/GynKreft/Data-EC/Nifti/EC548/regi...
304,555,20_t1_vibe_dixon_tra_p2__2min_W,/raid/erlend/GynKreft/Data-EC/Nifti/EC555/unre...,/raid/erlend/GynKreft/Data-EC/Nifti/EC555/regi...,/raid/erlend/GynKreft/Data-EC/Nifti/EC555/regi...
305,620,24_t1_vibe_dixon_tra_skra_p2_FOV_250__2min_W,/raid/erlend/GynKreft/Data-EC/Nifti/EC620/unre...,/raid/erlend/GynKreft/Data-EC/Nifti/EC620/regi...,/raid/erlend/GynKreft/Data-EC/Nifti/EC620/regi...


In [6]:
df = df[df.subj == 503].reset_index(drop=True)

In [8]:
# Define preprocessing function
#def preprocess_image(image_path, img_size):
#    transform = tio.Compose([
#        tio.ZNormalization(),
#        tio.CropOrPad(target_shape=(img_size, img_size, img_size))
#    ])
#    subject = tio.Subject(image=tio.ScalarImage(image_path))
#    transformed = transform(subject)
#    original_image = nib.load(image_path)  # Get original shape
#    crop_pad_transform = transform.transforms[-1]  # Save the CropOrPad object
#    return transformed['image'][tio.DATA], original_image  # Returns a numpy array

# Instantiate the model
model = monai_unet_model(dropout=0.2, in_channels=in_channels)

# Path to the saved weights
weights_path = os.path.join(prepathmodels, basename + '.pth')
print(weights_path)

# Load weights
model.load_state_dict(torch.load(weights_path, weights_only=True))
model.eval()  # Set the model to evaluation mode

# Preprocess the input image
img_size = 192  # Ensure this matches your model's input size
for i, subj in enumerate(df.subj):
    #image_path = df.imgpath1.iloc[i]
    
    # Get paths for all channels for the current subject
    image_paths = [df[col].iloc[i] for col in imgpath_list]
    print(image_paths)
    print(f"Subject {subj}")
    
    preprocessed_image, original_images = preprocess_images(image_paths, img_size)
    print(f'Shape of preprocessed image: {preprocessed_image.shape}')
    
    # Convert to tensor and add batch and channel dimensions
    input_tensor = preprocessed_image[None]
    print(f'Shape of input tensor: {input_tensor.shape}')
    
    with torch.no_grad():  # Disable gradient computation
        output_tensor = model(torch.tensor(input_tensor))  # Predict
        print(f'Shape of output tensor: {output_tensor.shape}')
        predicted_mask = torch.sigmoid(output_tensor).cpu().numpy()  # Apply sigmoid and convert to numpy
        print(f'Shape of predicted mask: {predicted_mask.shape}')
    
    # Wrap the array in a TorchIO ScalarImage
    wrapped_image = tio.ScalarImage(tensor=predicted_mask[0])
    
    # Define the transform (e.g., CropOrPad to original shape)
    transform = tio.CropOrPad(target_shape=original_images[0].shape)
    
    # Apply the transform
    transformed_image = transform(wrapped_image)
    
    # Extract the transformed array
    transformed_array = transformed_image[tio.DATA][0].numpy()

    # Plot the prediction
    show_max_mask_slices(original_images, transformed_array, df.subj.iloc[i], alpha=0.3)
    
    # Lagre bildet    
    save_path = os.path.join(prepathnifti, 'EC' + str(subj).zfill(3), 'analysis', 'PREDICTED-' + basename + '.nii.gz')
    print('Saving ' + save_path)
    mask_nifti = nib.Nifti1Image(transformed_array.astype(np.uint8), original_images[0].affine, header=original_images[0].header)
    nib.save(mask_nifti, save_path)

/raid/erlend/Dropbox/Precision_Imaging_in_Gynecologic_Cancer/EC/MonaiSegmentation/models/UNet-Monai-VIBE-T2-ADC-v0.1.pth
['/raid/erlend/GynKreft/Data-EC/Nifti/EC503/unregistered/vibe2min.nii.gz', '/raid/erlend/GynKreft/Data-EC/Nifti/EC503/registered/T2-2-vibe2min-header.nii.gz', '/raid/erlend/GynKreft/Data-EC/Nifti/EC503/registered/ADC-2-vibe2min-header.nii.gz']
Subject 503


RuntimeError: Standard deviation is 0 for masked values in image "image" (/raid/erlend/GynKreft/Data-EC/Nifti/EC503/registered/ADC-2-vibe2min-header.nii.gz)

In [ ]:
type(original_image)